# LiteraLevel · discursos de la Casa Blanca de Obama → Drive

Vídeos oficiales de obamawhitehouse.archives.gov, marcados como **dominio
público** (obra del gobierno federal de EE. UU.). Se bajan directos a
`MyDrive/literalevel_cara/casa_blanca/` y se miden con MediaPipe (blendshapes
ARKit, pose de la cabeza, tono y energía). Reanudable: si una sesión se cae,
se vuelve a ejecutar y sigue donde iba.

Antes: subir `lista.csv` (de `listar_casa_blanca.py`) con «Upload to Colab».

In [11]:
# --- 1. Drive y dependencias
# En VS Code, drive.mount() falla (MessageError request_auth): Drive se monta con
# Ctrl+Shift+P → «Colab: Mount Google Drive to Server...». En el navegador sí vale drive.mount.
import os
if not os.path.isdir('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        raise SystemExit('Monta Drive con «Colab: Mount Google Drive to Server...» y vuelve a ejecutar') from e
!pip -q install -U mediapipe librosa imageio-ffmpeg   # sin fijar: Colab ya va en Python 3.13
!wget -q -nc -O /content/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
import os, csv
DESTINO = '/content/drive/MyDrive/literalevel_cara/casa_blanca'
os.makedirs(f'{DESTINO}/videos', exist_ok=True); os.makedirs(f'{DESTINO}/medidas', exist_ok=True)
lista = [f for f in ['/content/lista.csv', f'{DESTINO}/lista.csv'] if os.path.exists(f)][0]
!cp -n {lista} {DESTINO}/lista.csv
filas = list(csv.DictReader(open(lista, encoding='utf8')))
TIPOS = {'discurso_semanal', 'presidente'}
MAX_VIDEOS = 160   # ~40-60 h; subirlo si hay sitio en Drive
sel = [f for f in filas if f['tipo'] in TIPOS and f['mp4'] and f['dominio_publico'] == '1']
# primero los discursos semanales (de frente, leyendo), luego el resto; tope para no llenar Drive
sel = sorted(sel, key=lambda f: (f['tipo'] != 'discurso_semanal', f['fecha']))[:MAX_VIDEOS]
print(len(filas), 'vídeos en la lista;', len(sel), 'seleccionados (discursos del presidente, dominio público)')

cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
2470 vídeos en la lista; 160 seleccionados (discursos del presidente, dominio público)


In [12]:
# --- 2. Bajar a Drive (varios a la vez; se salta lo ya bajado)
import concurrent.futures, subprocess
def bajar(f):
    nombre = f['fecha'] + '_' + f['slug'].rstrip('/').split('/')[-1][:80] + '.mp4'
    ruta = f'{DESTINO}/videos/{nombre}'
    if os.path.exists(ruta) and os.path.getsize(ruta) > 1e5:
        return 'ya'
    r = subprocess.run(['wget', '-q', '--tries=5', '-O', ruta + '.part', f['mp4']])
    if r.returncode == 0:
        os.replace(ruta + '.part', ruta); return 'ok'
    if os.path.exists(ruta + '.part'): os.remove(ruta + '.part')
    return 'fallo'
from collections import Counter
with concurrent.futures.ThreadPoolExecutor(6) as ex:
    res = Counter(ex.map(bajar, sel))
print(res)
!du -sh {DESTINO}/videos

Counter({'ya': 158, 'fallo': 2})
81G	/content/drive/MyDrive/literalevel_cara/casa_blanca/videos


In [13]:
MEDIR_PY = "\"\"\"Expresión y prosodia de cada fotograma de un clip de cara hablando.\n\nPor cada vídeo escribe un .npz con\n  t        segundos de cada fotograma\n  bs       las 52 blendshapes ARKit de MediaPipe (nombres en `bs_nombres`)\n  pose     giro de la cabeza en grados: guiñada, cabeceo, alabeo\n  ok       1 si se encontró cara\n  f0       tono en Hz (0 si no hay voz), llevado a los fotogramas\n  db       energía RMS en dB, llevada a los fotogramas\n\nEs la parte de `modelo_articulatorio/medir_video.py` que sirve para la vida\nde la cara (parpadeo, cejas, cabeza), más la pose y la prosodia. Si hay\nvarias caras se queda con la más grande. Reanudable.\n\nUso:\n  python medir_expresion.py --modelo face_landmarker.task --salida DIR clip1.mp4 [...]\n\"\"\"\nimport argparse\nimport os\n\n# librosa.pyin llama a ufuncs de numba en hilos; en Colab (Python 3.13) revienta con\n# señal 11 en discursos largos (visto el 25 sep con faulthandler). Un solo hilo lo evita.\nos.environ.setdefault('NUMBA_NUM_THREADS', '1')\nos.environ.setdefault('NUMBA_THREADING_LAYER', 'workqueue')\nimport subprocess\nimport tempfile\n\nimport cv2\nimport imageio_ffmpeg\nimport librosa\nimport mediapipe as mp\nimport numpy as np\nfrom mediapipe.tasks import python as mp_python\nfrom mediapipe.tasks.python import vision\n\n\ndef pose_de_matriz(m):\n    \"\"\"Ángulos (guiñada, cabeceo, alabeo) en grados de la matriz 4x4 de MediaPipe.\"\"\"\n    r = np.asarray(m)[:3, :3]\n    cabeceo = np.degrees(np.arctan2(r[2, 1], r[2, 2]))\n    guinada = np.degrees(np.arcsin(-np.clip(r[2, 0], -1, 1)))\n    alabeo = np.degrees(np.arctan2(r[1, 0], r[0, 0]))\n    return guinada, cabeceo, alabeo\n\n\ndef tiene_video(ruta):\n    \"\"\"Algunos tramos bajan solo con audio; cv2 se cuelga con ellos (visto el 25 sep).\"\"\"\n    r = subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), '-hide_banner', '-i', ruta],\n                       capture_output=True, text=True, errors='replace')\n    return 'Video:' in r.stderr\n\n\ndef prosodia(video, t):\n    with tempfile.TemporaryDirectory() as d:\n        wav = os.path.join(d, 'a.wav')\n        subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), '-loglevel', 'error', '-y', '-i', video,\n                        '-ac', '1', '-ar', '16000', wav], check=True)\n        y, sr = librosa.load(wav, sr=16000)\n    salto = 160  # 10 ms\n    # pyin por trozos de 60 s: sobre un discurso de una hora entero pide decenas de GB\n    # (matrices de probabilidad por fotograma × tono) y el sistema mata el proceso.\n    trozo = 60 * sr\n    f0s, vozs = [], []\n    for ini in range(0, len(y), trozo):\n        seg = y[ini:ini + trozo]\n        if len(seg) < 2048:\n            n = int(np.ceil(len(seg) / salto))\n            f0s.append(np.full(n, np.nan)); vozs.append(np.zeros(n, bool))\n            continue\n        f, v, _ = librosa.pyin(seg, fmin=65, fmax=400, sr=sr, frame_length=1024, hop_length=salto)\n        n = int(np.ceil(len(seg) / salto))          # fotogramas propios del trozo, sin el de cierre\n        f0s.append(f[:n]); vozs.append(v[:n])\n    f0, voz = np.concatenate(f0s), np.concatenate(vozs)\n    f0 = np.where(voz, f0, 0.0)\n    rms = librosa.feature.rms(y=y, frame_length=400, hop_length=salto)[0]\n    db = 20 * np.log10(rms + 1e-5)\n    ta = np.arange(len(f0)) * salto / sr\n    tr = np.arange(len(db)) * salto / sr\n    # f0 al fotograma: el valor más cercano (no interpolar entre voz y silencio)\n    idx = np.clip(np.searchsorted(ta, t), 0, len(f0) - 1)\n    return f0[idx], np.interp(t, tr, db)\n\n\nclass LectorFfmpeg:\n    \"\"\"Fotogramas ya reducidos a `ancho` px y a `fps_max`: ffmpeg decodifica y escala\n    con varios hilos, y MediaPipe recibe imágenes pequeñas. Con vídeo HD es 2-3 veces\n    más rápido que leer con cv2 a tamaño completo. Devuelve BGR, como cv2.\"\"\"\n\n    def __init__(self, video, ancho, fps_max=25):\n        cap = cv2.VideoCapture(video)\n        w, h = cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT)\n        fps = cap.get(cv2.CAP_PROP_FPS) or 25\n        cap.release()\n        self.w = int(min(ancho, w)) // 2 * 2\n        self.h = int(round(h * self.w / w / 2)) * 2\n        self.fps = min(fps, fps_max)\n        self.p = subprocess.Popen([imageio_ffmpeg.get_ffmpeg_exe(), '-loglevel', 'error', '-i', video,\n                                   '-vf', f'fps={self.fps},scale={self.w}:{self.h}', '-f', 'rawvideo',\n                                   '-pix_fmt', 'bgr24', '-'], stdout=subprocess.PIPE)\n        self.n = self.w * self.h * 3\n\n    def get(self, prop):\n        return self.fps\n\n    def read(self):\n        b = self.p.stdout.read(self.n)\n        if len(b) < self.n:\n            return False, None\n        return True, np.frombuffer(b, np.uint8).reshape(self.h, self.w, 3)\n\n    def release(self):\n        self.p.stdout.close()\n        self.p.wait()\n\n\ndef medir(video, lmk, salida, ancho=0):\n    cap = LectorFfmpeg(video, ancho) if ancho else cv2.VideoCapture(video)\n    fps = cap.get(cv2.CAP_PROP_FPS) or 25\n    t, bs, pose, ok = [], [], [], []\n    nombres = None\n    i = 0\n    while True:\n        bien, img = cap.read()\n        if not bien:\n            break\n        ts = int(i * 1000 / fps)\n        r = lmk.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB,\n                                          data=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)), ts)\n        t.append(i / fps)\n        if r.face_blendshapes:\n            # la cara más grande: la de mayor anchura de la malla\n            k = int(np.argmax([np.ptp([p.x for p in c]) for c in r.face_landmarks]))\n            cats = r.face_blendshapes[k]\n            nombres = nombres or [c.category_name for c in cats]\n            bs.append([c.score for c in cats])\n            pose.append(pose_de_matriz(r.facial_transformation_matrixes[k]))\n            ok.append(1)\n        else:\n            bs.append([0.0] * 52)\n            pose.append((0.0, 0.0, 0.0))\n            ok.append(0)\n        i += 1\n    cap.release()\n    t = np.array(t)\n    f0, db = prosodia(video, t)\n    np.savez_compressed(salida, t=t, bs=np.array(bs, np.float32), pose=np.array(pose, np.float32),\n                        ok=np.array(ok, np.uint8), f0=f0.astype(np.float32), db=db.astype(np.float32),\n                        bs_nombres=np.array(nombres or []), fps=fps)\n    return len(t), float(np.mean(ok)) if ok else 0.0\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--modelo', required=True)\n    ap.add_argument('--salida', required=True)\n    ap.add_argument('--ancho', type=int, default=0,\n                    help='reducir el vídeo a este ancho (p. ej. 640) antes de medir; 0 = tamaño original')\n    ap.add_argument('videos', nargs='+')\n    a = ap.parse_args()\n    os.makedirs(a.salida, exist_ok=True)\n    for v in a.videos:\n        destino = os.path.join(a.salida, os.path.splitext(os.path.basename(v))[0] + '.npz')\n        if os.path.exists(destino):\n            continue\n        if not tiene_video(v):\n            print(f'{os.path.basename(v)}: sin imagen, se salta')\n            continue\n        opts = vision.FaceLandmarkerOptions(\n            base_options=mp_python.BaseOptions(model_asset_path=a.modelo),\n            running_mode=vision.RunningMode.VIDEO, num_faces=2,\n            output_face_blendshapes=True, output_facial_transformation_matrixes=True)\n        with vision.FaceLandmarker.create_from_options(opts) as lmk:\n            n, frac = medir(v, lmk, destino, a.ancho)\n        print(f'{os.path.basename(v)}: {n} fotogramas, cara en {frac:.0%}')\n\n\nif __name__ == '__main__':\n    main()\n"
# --- 3. Preparar la medición (no la lanza: se lanza desde la terminal del servidor, que no depende del kernel)
# Los procesos hijos del kernel mueren cuando VS Code reconecta el kernel (visto dos veces el 25 sep).
open('/content/medir_expresion.py', 'w', encoding='utf8').write(MEDIR_PY)
import glob
videos = sorted(glob.glob(f'{DESTINO}/videos/*.mp4'), key=lambda v: ('weekly-address' not in v, os.path.getsize(v)))
faltan = [v for v in videos if not os.path.exists(f"{DESTINO}/medidas/{os.path.basename(v)[:-4]}.npz")]
NPROC = min(6, os.cpu_count() or 2)   # 8 a la vez llenaron los 51 GB de RAM
NL = chr(10)
lineas = ['#!/bin/bash', 'pkill -f "^python -u /content/medir_expresion" ; sleep 1']
for i in range(NPROC):
    t = faltan[i::NPROC]
    if not t: continue
    open(f'/content/lista_{i}.txt', 'w').write(NL.join(t) + NL)
    # un proceso por vídeo: si MediaPipe revienta (señal 11, visto el 25 sep) solo se pierde ese vídeo
    lineas.append(f'(while IFS= read -r v; do python -u /content/medir_expresion.py '
                  f'--modelo /content/face_landmarker.task --salida "{DESTINO}/medidas" --ancho 640 "$v" '
                  f'|| echo "$v" >> /content/fallidos.txt; done < /content/lista_{i}.txt) '
                  f'> /content/medir_{i}.log 2>&1 &')
lineas.append('wait')
open('/content/lanzar.sh', 'w').write(NL.join(lineas) + NL)
print(len(videos), 'vídeos,', len(faltan), 'por medir,', NPROC, 'procesos')
print('Ahora, en la terminal del servidor (Colab: Open Terminal):')
print('  nohup setsid bash /content/lanzar.sh > /content/lanzar.log 2>&1 &')

158 vídeos, 34 por medir, 6 procesos
Ahora, en la terminal del servidor (Colab: Open Terminal):
  nohup setsid bash /content/lanzar.sh > /content/lanzar.log 2>&1 &


In [14]:
# --- 4. Progreso (ejecutar cuando quieras; no para nada)
import glob, subprocess
hechos = len(glob.glob(f'{DESTINO}/medidas/*.npz')); total = len(glob.glob(f'{DESTINO}/videos/*.mp4'))
b = int(20 * hechos / max(total, 1))
vivos = subprocess.run("pgrep -f '^python -u /content/medir_expresion' | wc -l", shell=True, capture_output=True, text=True).stdout.strip()
print('█' * b + '░' * (20 - b), f'{hechos}/{total}', f'· procesos vivos: {vivos}')
!grep -h -v -E '^(W0000|I0000|WARNING|INFO)|inference_feedback|XNNPACK' /content/medir_*.log 2>/dev/null | tail -n 6
!dmesg 2>/dev/null | grep -i -E 'killed process|out of memory' | tail -n 3

███████████████░░░░░ 124/158 · procesos vivos: 0
